# 03 — ET PP+HP Two-Stage Stage 1 분류 (스태킹 다양성용)

기존 `clf/et.ipynb`(HP만 흔드는 분류 노트북)의 **쌍둥이** — 여기에 **PP 6축을 trial 축에 추가**.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/03_two_stage/default/clf/et/pphp/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **PP**: 6축 Optuna 탐색 ([pp_hp_strategy.md §3](../../../../pp_hp_strategy.md)). 나머지는 PP_FIXED와 동일하게 고정.
- **HP**: hp-only(`clf/et.ipynb`)와 **동일 범위** — `models.get_clf_search_space('et')`. anchor 없음 (기존 clf 노트북과 동일 — 1차 분류 1 trial 신뢰도 낮음).
- **objective**: 분류기 calibration이 unit RMSE에 미치는 영향을 직접 평가 — `unit_RMSE(unit_mean(die_proba) × E[Y|Y>0])`.
- **CV**: hp-only와 동일 — unit-level 5-fold KFold. clf 단독은 후처리 없음 (combine 단계에서 reg와 곱한 뒤 후처리).

> ⚠ pp+hp는 trial마다 전처리를 다시 도므로 hp-only보다 느리다.

## 1. 환경 설정 + 모듈 import

In [ ]:
import os, sys

# Google Drive 파일 ID들 — Colab에서 코드/데이터/모듈 zip을 자동으로 받아 풀 때 사용 (로컬은 무시)
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip = setup.py + utils/
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'  # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'  # preprocessing.zip = cleaning/outlier/scaling 등
GDRIVE_MODELING_ID      = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # modeling.zip = 3_modeling/modules (코드 수정 시 재업로드)
GDRIVE_OUTPUT_ID       = '1ts73qEMmjX8cKIb-QeDQ-TMeyudFGWzs'  # 4_output.zip = 기존 실험 산출물 (RESUME용)
RESUME                 = True   # True=기존 optuna db에 trial 이어 붙임 / False=처음부터 (db 있으면 의도적 에러)

# Colab이면 필요한 zip들을 받아 풀고(이미 풀려 있으면 skip), 로컬이면 ../../../..만
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/modules/pp_hpo.py'):
        assert GDRIVE_MODELING_ID, 'GDRIVE_MODELING_ID가 비어있음 — modules.zip Drive ID 입력 필요'
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modules.zip')
        os.system('unzip -qo /content/modules.zip -d /content/project/3_modeling')
    if RESUME and GDRIVE_OUTPUT_ID and not os.path.exists('/content/project/4_output/01_zit'):
        os.system(f'gdown {GDRIVE_OUTPUT_ID} -O /content/4_output.zip')
        os.system('unzip -qo /content/4_output.zip -d /content/project')
        os.remove('/content/4_output.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈(2_preprocessing) 경로 + `from modules import ...` 가 3_modeling/modules를 찾게
PREP_ROOT = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PREP_ROOT not in sys.path:
    sys.path.insert(0, PREP_ROOT)
MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from modules import hpo, models, pp_hpo   # pp_hpo: PP를 trial 축에 넣는 래퍼 (run_pp_hpo / run_pp_clf_hpo / refit_pp_best / make_cached_preprocess)
from meta_features import add_meta_features   # (pp_hpo.make_cached_preprocess가 내부에서 사용 — import 가능 확인용)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Available models: {models.AVAILABLE_MODELS} | clf: {models.CLF_AVAILABLE_MODELS}')

## 2. 실험 설정

In [ ]:
# 분류 모델 고정 (Two-Stage Stage 1 — y>0인 unit 식별, PP+HP 동시 탐색)
CLF_MODEL_NAME = 'et'
assert CLF_MODEL_NAME in models.CLF_AVAILABLE_MODELS

EXP_ID   = 'ts-clf-et-pphp'
EXP_MEMO = 'PP+HP 동시 탐색 (스태킹 다양성). HP 범위·CV는 clf/et.ipynb와 동일, PP만 6축 추가, anchor 없음'
USER     = 'jh'

N_TRIALS         = 3000
N_FOLDS          = 5
N_STARTUP_TRIALS = 50
N_JOBS           = -1   # 모델 학습 병렬도 (-1 = 전체 코어; catboost는 thread_count로 매핑)
TIMEOUT_SEC      = 20 * 60 * 60  # 초 단위, None=무제한 (Colab 타임아웃 대비)

CLIP_Y_EXTREME = True   # train y의 max(1.0, 1건)를 두 번째 큰 값으로 clip
PP_CACHE_SIZE  = 2      # cached_preprocess가 보관할 PP 조합 개수 — 메모리 보고 조정

OUT_DIR = os.path.join(OUTPUT_DIR, '03_two_stage', 'default', 'clf', 'et', 'pphp')
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

print(f'EXP: {EXP_ID} | USER: {USER} | CLF_MODEL_NAME: {CLF_MODEL_NAME}')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS} | N_JOBS={N_JOBS} | TIMEOUT_SEC={TIMEOUT_SEC} | PP_CACHE_SIZE={PP_CACHE_SIZE}')
print(f'OUT_DIR={OUT_DIR}')

## 3. 데이터 로드 + target clip

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

# train y의 극단값(1.0, 1건)만 두 번째로 큰 값으로 clip — 학습 입력 안정화 (원본 ys는 보존)
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# 트리 모델은 target 변환이 결과를 거의 안 바꿔서 'none'으로 고정 (clf는 안 씀 — 둘 다 None)
target_transform_fn = None
target_inverse_fn   = None
print(f'[target transform] none (strategy_common §24 — 트리 target_transform=none 통일)')

## 4. 전처리 캐시 준비 (PP는 Optuna 6축 — pp_hpo)

In [ ]:
# PP는 더 이상 고정값이 아니라 Optuna 탐색 6축 (pp_hpo.PP_SEARCH_CANDIDATES — pp_hp_strategy.md §3)
print('[PP 탐색 6축]')
for k, v in pp_hpo.PP_SEARCH_CANDIDATES.items():
    print(f'  {k:28s} {v}')
print('  (고정: corr_keep_by=std, post_impute_corr_keep_by=std, const_threshold=1e-6, remove_duplicates=True,\n'
      '         imputation_method=spatial, outlier=winsorize 0.0/0.99, Stage0 EXCLUDE_COLS)')

# (xs, ys, feat_cols, xs_dict) 고정 → trial마다 호출하는 캐시된 전처리 함수.
# preprocess.run(Stage0→cleaning→winsorize) + add_meta_features(position='raw', die_xy)를 PP 조합별로 캐시.
cached_prep = pp_hpo.make_cached_preprocess(
    xs, ys_input, feat_cols, xs_dict,
    position_mode='raw', use_die_xy=True,
    maxsize=PP_CACHE_SIZE, suppress_stdout=True,
)
print(f'\n[cached_preprocess 준비 완료] maxsize={PP_CACHE_SIZE}')

## 5. Optuna PP+HP CLF HPO (PP 6축 + HP wide range, anchor 없음)

In [ ]:
study_meta = {
    'exp_id':               EXP_ID,
    'exp_memo':             EXP_MEMO,
    'user':                 USER,
    'clf_model_name':       CLF_MODEL_NAME,
    'clip_y_extreme':       CLIP_Y_EXTREME,
    'pp_search_candidates': pp_hpo.PP_SEARCH_CANDIDATES,
    'pp_cache_size':        PP_CACHE_SIZE,
    'n_trials':             N_TRIALS,
    'n_folds':              N_FOLDS,
    'n_jobs':               N_JOBS,
    'n_startup_trials':     N_STARTUP_TRIALS,
    'timeout_sec':          TIMEOUT_SEC,
    'seed':                 SEED,
    'sampler':              'TPE seed=None multivariate group',
    'pruner':               'MedianPruner n_warmup=10',
}

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# 분류 HPO: 매 trial PP 6축 + HP 샘플 → die-level 학습 + unit 평균 → ×E[Y|Y>0] → unit RMSE를 minimize.
# anchor 없음 (clf 노트북과 동일). val/test RMSE는 매 trial user_attr.
res = pp_hpo.run_pp_clf_hpo(
    cached_prep,
    xs_dict['train'], ys_input['train'],
    CLF_MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    resume_study=RESUME,
    user_attrs=study_meta,
    sampler=TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS),
    pruner=MedianPruner(n_warmup_steps=10),
    enqueue_trials=None,         # anchor enqueue 안 함 (clf 노트북과 동일)
    timeout=TIMEOUT_SEC,
    xs_val_raw=xs_dict['validation'], ys_val_unit=ys_input['validation'],
    xs_test_raw=xs_dict['test'],       ys_test_unit=ys_input['test'],
)
study       = res['study']
best_params = res['best_params']
y_pos_const = res['y_pos_const']   # E[Y | Y>0] — Stage 2 회귀를 "상수"로 대체한 값 (combine 전까지 이걸로 평가)

print(f'\n[HPO 완료] best train RMSE = {res["best_value"]:.6f}')
print(f'y_pos_const (E[Y|Y>0]) = {y_pos_const:.6f}')
print(f'[PP 캐시] {cached_prep.counters}')
print(f'best_params = {best_params}')

## 6. Best trial 재학습 (best PP로 전처리 재실행 + K-fold OOF)

In [ ]:
# best PP로 preprocess 재실행 → 그 데이터로 best HP 5-fold 재학습 (refit_pp_clf_best가 hpo.refit_clf_best 위임).
out = pp_hpo.refit_pp_clf_best(
    cached_prep, xs_dict, ys_input,
    CLF_MODEL_NAME, best_params,
    n_folds=N_FOLDS, n_jobs=N_JOBS,
)
final           = out['refit_result']
xs_train        = out['xs_train']
xs_val          = out['xs_val']
xs_test         = out['xs_test']
feat_cols_clean = out['feat_cols']
study_meta['effective_pp_params'] = out['effective_pp_params']
study_meta['best_pp_params']      = out['pp_params']
print(f"[best PP] {out['pp_params']}")
print(f"[best PP 전처리 후 feat_cols] {len(feat_cols_clean)}")
print(f"[PP 캐시 최종] {cached_prep.counters}")

def _unit_mean_proba(xs_split, die_proba):
    df = pd.DataFrame({KEY_COL: xs_split[KEY_COL].values, 'p': die_proba})
    return df.groupby(KEY_COL, sort=False)['p'].mean()

def _rmse(pred_unit, y_unit_df):
    aligned = pred_unit.loc[y_unit_df.set_index(KEY_COL).index]
    return float(np.sqrt(np.mean((aligned.values - y_unit_df[TARGET_COL].values) ** 2)))

oof_unit_proba  = _unit_mean_proba(xs_train, final['oof_proba_die'])
val_unit_proba  = _unit_mean_proba(xs_val,   final['val_proba_die'])
test_unit_proba = _unit_mean_proba(xs_test,  final['test_proba_die'])

oof_rmse  = _rmse(oof_unit_proba * y_pos_const, ys_input['train'])
val_rmse  = _rmse(val_unit_proba * y_pos_const, ys_input['validation'])
test_rmse = _rmse(test_unit_proba * y_pos_const, ys_input['test'])

print(f'\n[Refit 완료] (clf 단독, prob × y_pos_const)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')

## 7. 산출물 저장 (clf 단독 — 후처리 없음)

In [ ]:
# clf 산출물 저장: oof/val/test die.csv(확률) + unit.csv(평균확률·pred) + fold_models.pkl + best_params.json.
# (clf 단독은 후처리 없음 — combine 단계에서 reg 예측과 곱한 뒤 적용)
hpo.save_clf_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    y_pos_const=y_pos_const,
    study_meta=study_meta,
)

for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:30s}  {sz:>10,.1f} KB')

# Colab이면 산출물을 zip으로 묶어 로컬 PC로 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', f'clf_{CLF_MODEL_NAME}_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크')
        display(FileLink(_zip))
except ImportError:
    pass